# 02 · Real-time fraud detection on top of Customer 360 — Lumora Bank (fictional)
Transactions arrive per source-system account. The golden-record crosswalk links every account to one customer, so the
model sees each customer's full behavior. Scoring blends interpretable rules with a logistic-regression model; the
top alerts get a GenAI case narrative for the investigator. Run notebook 01 first (or this notebook rebuilds the 360).

In [ ]:
# ---- CONFIG ----
DATA_DIR  = "../data/sample"        # local folder, or an S3 prefix e.g. "s3://<bucket>/<prefix>/data"
PUBLISH   = False                   # True = write tables to the Glue Data Catalog (Iceberg via Athena)
DATABASE  = "<glue_database>"
S3_PATH   = "s3://<bucket>/<prefix>"  # no trailing slash
WORKGROUP = None                    # SageMaker Unified Studio: set to the project's Athena workgroup
REGION    = "us-east-1"
MODEL_ID  = "us.amazon.nova-pro-v1:0"   # any Bedrock text model enabled in REGION
# ----------------

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))          # repo layout; in SMUS upload src/ next to notebooks/
import pandas as pd
from lumora import c360, fraud, genai, aws_io
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 200)
rd = lambda n: aws_io.read_csv(f"{DATA_DIR}/{n}.csv")

## 1 · Customer 360 crosswalk + transaction stream

In [ ]:
res = c360.run(rd("core_banking_customers"), rd("crm_contacts"), rd("cards_accounts"))
gold, xref = res["gold"], res["xref"]
txns = rd("transactions").astype({"amount": float, "is_fraud": int})
print(f"{len(txns)} transactions across {txns.account_ref.nunique()} accounts → {gold.customer_id.nunique()} customers")
display(txns.head())

## 2 · Behavioral features (per customer, not per account)

In [ ]:
feats = fraud.build_features(txns, xref)
display(feats[["customer_id","ts","amount"] + fraud.FEATURES].sample(5, random_state=3))

## 3 · Train on earlier days, test on later days

In [ ]:
model, scored, metrics = fraud.train_and_score(feats)
display(pd.Series(metrics, name="metric"))
print("\nWhat the model learned (standardized weights):"); display(model.weights().round(2))
print(scored.decision.value_counts())

## 4 · The alert queue

In [ ]:
alerts = (scored[scored.decision != "APPROVE"].merge(gold[["customer_id","name_en","segment"]], on="customer_id")
          .sort_values("fraud_score", ascending=False))
display(alerts[["ts","customer_id","name_en","amount","channel","merchant_country","device_id",
                "txn_count_1h","fraud_score","decision","is_fraud"]].head(15))

## 5 · Score one incoming transaction (what a stream consumer would call)

In [ ]:
row = alerts.iloc[0][fraud.FEATURES].to_dict()
fraud.score_one(model, row)

## 6 · GenAI case narrative for the top alert (Amazon Bedrock)

In [ ]:
top_cust = alerts.iloc[0].customer_id
case = alerts[alerts.customer_id == top_cust][["ts","amount","channel","merchant_category","merchant_country",
                                              "city","device_id","fraud_score","decision"]].to_dict("records")
customer = {k: (None if pd.isna(v) else v) for k, v in gold[gold.customer_id == top_cust].iloc[0].items()}
print(genai.fraud_case_narrative(customer, case, model_id=MODEL_ID, region=REGION))

## 7 · Publish (optional)

In [ ]:
os.makedirs("../output", exist_ok=True); scored.to_csv("../output/fraud_scored_txns.csv", index=False)
if PUBLISH:
    aws_io.publish({"fraud_scored_txns": scored.astype({"ts": "string"})}, DATABASE, S3_PATH, WORKGROUP)